# Practice 05 — NLP & Transformers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/05-nlp-transformers.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/05-nlp-transformers.ipynb)

Companion drills for **Phase 5** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with [TF-IDF](https://learn-by-visuallization.org/illustrated/5-nlp-transformers/tfidf.html),
[word2vec](https://learn-by-visuallization.org/illustrated/5-nlp-transformers/word2vec.html) and the scrub-through
[self-attention explainer](https://learn-by-visuallization.org/illustrated/5-nlp-transformers/self-attention.html).

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup, no GPU needed).

In [ ]:
NEEDED = [("numpy", "numpy"), ("sklearn", "scikit-learn")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
rng = np.random.default_rng(0)


### Exercise 1 — TF-IDF by hand
For the 3-document corpus, compute the **smoothed IDF** of each vocabulary word:
`idf(w) = ln((1+N)/(1+df(w))) + 1` (sklearn's formula). Store a dict `idf`.

*Hint: df(w) = number of DOCUMENTS containing w (not total occurrences).*

In [ ]:
docs = ["the cat sat", "the dog sat", "the cat ate the fish"]
vocab = sorted(set(" ".join(docs).split()))

idf = None  # TODO: {word: idf value}


In [ ]:
import math
def _idf(w):
    df = sum(w in d.split() for d in docs)
    return math.log((1 + 3) / (1 + df)) + 1
check("idf matches sklearn's smoothed formula for every word",
      lambda: all(abs(idf[w] - _idf(w)) < 1e-9 for w in vocab),
      'df counts documents containing the word; "the" (df=3) gets the lowest idf')
check("'the' is the least informative word",
      lambda: min(idf, key=idf.get) == "the",
      "appearing everywhere = carrying no signal — exactly what idf downweights")

### Exercise 2 — cosine ranking with real TF-IDF
Use sklearn's `TfidfVectorizer` on `news` and rank the documents against the query
`"team wins the final"`. Store `ranking`: document indices, best match first.

*Hint: `vec.fit_transform(news)`, `vec.transform([query])`, then `cosine_similarity` and `argsort()[::-1]`.*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

news = [
    "the home team wins the championship final in extra time",
    "central bank raises interest rates amid inflation fears",
    "star striker injured before the cup final, team hopeful",
    "new species of frog discovered in the rainforest",
]
query = "team wins the final"

ranking = None  # TODO: indices of news, most similar first


In [ ]:
check("sports stories outrank finance & frogs",
      lambda: list(ranking)[:2] in ([0, 2], [2, 0]) and list(ranking)[0] == 0,
      "doc 0 shares team/wins/final; doc 2 shares team/final — both beat rates and frogs")

### Exercise 3 — scaled dot-product attention, the formula itself
Implement `attention(Q, K, V)` = `softmax(Q·Kᵀ/√d)·V` for matrices (n×d). This is THE transformer
equation — the one you scrubbed step-by-step in the explainer.

*Hint: scores `Q @ K.T / np.sqrt(d)`; row-wise softmax (subtract each row's max first); then `@ V`.*

In [ ]:
def attention(Q, K, V):
    # TODO: return (output, weights)
    ...


In [ ]:
_Q = np.array([[1., 0.], [0., 1.]])
_K = np.array([[1., 0.], [0., 1.]])
_V = np.array([[10., 0.], [0., 10.]])
try:
    out, W = attention(_Q, _K, _V)
except Exception:
    out, W = None, None
check("weights are a distribution (rows sum to 1)",
      lambda: np.allclose(W.sum(1), 1), "softmax per ROW (each query token mixes to 1)")
check("each query attends most to its matching key",
      lambda: W[0, 0] > W[0, 1] and W[1, 1] > W[1, 0],
      "Q·Kᵀ is largest where query aligns with key")
check("output is the weighted Value blend",
      lambda: np.allclose(out, W @ _V), "out = weights @ V — nothing more")

### Exercise 4 — why order needs positional encoding
`emb` maps two sentences with the SAME words in different order to token vectors. Show that
**mean-pooled bag-of-vectors cannot tell them apart**: compute both pooled vectors and their
difference norm `gap` (it will be ~0). That failure is the reason positional encodings exist.

*Hint: `E[s.split()]` … actually build `np.mean([emb[w] for w in sentence.split()], axis=0)`.*

In [ ]:
emb = {w: rng.normal(size=4) for w in "man bites dog".split()}
s1, s2 = "man bites dog", "dog bites man"

pool1 = None  # TODO mean of word vectors of s1
pool2 = None  # TODO mean of word vectors of s2
gap   = None  # TODO np.linalg.norm(pool1 - pool2)


In [ ]:
check("order-blind pooling: gap is ~0",
      lambda: gap is not None and gap < 1e-12,
      "same multiset of words -> identical mean. Attention alone has the same blindness — "
      "positional encoding is the fix")

### Stretch goal — sinusoidal positions
Implement the classic `PE[pos, 2i] = sin(pos/10000^(2i/d))`, `PE[pos, 2i+1] = cos(...)` matrix for
d=16, 20 positions, and plot it with matplotlib. Compare with the
[positional-encoding explainer](https://learn-by-visuallization.org/illustrated/5-nlp-transformers/positional-encoding.html).

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")
